# Part II - Comparing flight delays by airport and by airline
## by Michelle Sollicito


## Investigation Overview


> This presentation has the following goals:
- illustrate departure and arrival delays at four airports
- illustrate departure and arrival delays of United Airlines vs Delta



## Dataset Overview and Executive Summary

> Dataset used:
- January 2026 flight delay data
- Downloaded from https://www.transtats.bts.gov/Fields.asp?gnoyr_VQ=FGJ


> **Rubric Tip**: The key insights can be in a bulleted or numbered list for ease of review.

In [1]:
# import all packages and set plots to be embedded inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# suppress warnings from final output
import warnings
warnings.simplefilter("ignore")

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
#import the file

df = pd.read_csv('T_ONTIME_REPORTING.csv')

## (Visualization 1)

> Visualization 1 shows the positive relationship between delay departure minutes and arrival departure minutes.  So from this we can deduce that if flights are delayed on departure they will experience similar levels of arrival delays.



In [3]:
# Take a sample to prevent overplotting

cleanup3 = df[~((df['CANCELLED'] == 0) & (df['ARR_DELAY_NEW'].isna()))]
scatter_data = cleanup3[
    ['DEP_DELAY', 'ARR_DELAY_NEW']
].dropna().sample(
    10000,
    random_state=42
)

plt.figure(figsize=(8, 6))

sns.scatterplot(
    data=scatter_data,
    x='DEP_DELAY',
    y='ARR_DELAY_NEW',
    alpha=0.3
)

plt.xlabel('Departure Delay (minutes)')
plt.ylabel('Arrival Delay (minutes)')
plt.title('Relationship Between Departure and Arrival Delays')

plt.show()

<class 'NameError'>: name 'df' is not defined

## (Visualization 2)

> Visualization 2 illustrates that generally, airports have small delays with some right skew for outlier longer delays.  FLL stands out as it has higher delays and also higher variability than ATL, LAX and LGA.

In [4]:
airports = ['ATL', 'FLL', 'LAX', 'LGA']

box_data = cleanup3[
    cleanup3['ORIGIN'].isin(airports)
].copy()

# Restrict display to the 99th percentile so extreme
# outliers do not make the main distributions unreadable
limit = box_data['DEP_DELAY'].quantile(0.99)

box_data = box_data[
    box_data['DEP_DELAY'] <= limit
]

plt.figure(figsize=(8, 5))

sns.boxplot(
    data=box_data,
    x='ORIGIN',
    y='DEP_DELAY'
)

plt.xlabel('Origin Airport')
plt.ylabel('Departure Delay (minutes)')
plt.title('Departure Delay by Origin Airport')

plt.show()



<class 'NameError'>: name 'cleanup3' is not defined

## (Visualization 3)

> Visualization 3 clearly illustrates that Delta overall delays in minutes are much higher than United Airlines delays.  But this must be taken in context, see Visualization 4.

In [5]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# The two airlines
airlines = ['UA', 'DL']

# Filter to United and Delta
plot_data = cleanup3[
    cleanup3['OP_UNIQUE_CARRIER'].isin(airlines)
].copy()

# Create a combined dataframe for departure and arrival delays
departure = plot_data[['OP_UNIQUE_CARRIER', 'DEP_DELAY']].copy()
departure['Delay_Type'] = 'Departure'
departure = departure.rename(columns={'DEP_DELAY': 'Delay'})

arrival = plot_data[['OP_UNIQUE_CARRIER', 'ARR_DELAY_NEW']].copy()
arrival['Delay_Type'] = 'Arrival'
arrival = arrival.rename(columns={'ARR_DELAY_NEW': 'Delay'})

# Combine the two datasets
facet_data = pd.concat([departure, arrival], ignore_index=True)

# Set bin size
binsize = 50

# Create common bins for both departure and arrival
bins = np.arange(
    facet_data['Delay'].min(),
    facet_data['Delay'].max() + binsize,
    binsize
)

# Create one FacetGrid
g = sns.FacetGrid(
    facet_data,
    row='Delay_Type',
    col='OP_UNIQUE_CARRIER',
    col_order=airlines,
    height=3,
    aspect=1.2,
    sharex=True,
    sharey=True
)

# Add histograms
g.map(
    plt.hist,
    'Delay',
    bins=bins,
    log=True,
    color='steelblue',
    edgecolor='black'
)

# Labels
g.set_axis_labels("Delay (minutes)", "Flights")
g.set_titles("{row_name} — {col_name}")

# Overall title
plt.subplots_adjust(top=0.88)
g.fig.suptitle(
    "Departure and Arrival Delay Distributions by Airline"
)

plt.show()

<class 'ModuleNotFoundError'>: No module named 'pandas'

## (Visualization 4)

> Visualization 4 clearly illustrates that Delta delay RATES are much higher than United Airlines delay rates.  

In [6]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# =======================================================
# UNITED vs DELTA - DEPARTURE AND ARRIVAL DELAY RATES
# =======================================================

airlines = ['UA', 'DL']

# -------------------------------------------------------
# 1. Filter to United and Delta
# -------------------------------------------------------

plot_data = cleanup3[
    cleanup3['OP_UNIQUE_CARRIER'].isin(airlines)
].copy()


# -------------------------------------------------------
# 2. Calculate DEPARTURE delay rates
# -------------------------------------------------------

departure = plot_data[
    ['OP_UNIQUE_CARRIER', 'DEP_DELAY']
].dropna().copy()

# 15 minutes or more = delayed
departure['Delayed'] = departure['DEP_DELAY'] >= 15

departure_rates = (
    departure
    .groupby('OP_UNIQUE_CARRIER')
    .agg(
        Total_Flights=('Delayed', 'size'),
        Delayed_Flights=('Delayed', 'sum')
    )
    .reset_index()
)

departure_rates['Delay_Rate'] = (
    departure_rates['Delayed_Flights']
    / departure_rates['Total_Flights']
    * 100
)

departure_rates['Delay_Type'] = 'Departure'


# -------------------------------------------------------
# 3. Calculate ARRIVAL delay rates
# -------------------------------------------------------

arrival = plot_data[
    ['OP_UNIQUE_CARRIER', 'ARR_DELAY_NEW']
].dropna().copy()

# 15 minutes or more = delayed
arrival['Delayed'] = arrival['ARR_DELAY_NEW'] >= 15

arrival_rates = (
    arrival
    .groupby('OP_UNIQUE_CARRIER')
    .agg(
        Total_Flights=('Delayed', 'size'),
        Delayed_Flights=('Delayed', 'sum')
    )
    .reset_index()
)

arrival_rates['Delay_Rate'] = (
    arrival_rates['Delayed_Flights']
    / arrival_rates['Total_Flights']
    * 100
)

arrival_rates['Delay_Type'] = 'Arrival'


# -------------------------------------------------------
# 4. Combine departure and arrival results
# -------------------------------------------------------

delay_rates = pd.concat(
    [departure_rates, arrival_rates],
    ignore_index=True
)

# Add a constant x value.
# We do NOT need airline as the x-axis because airline
# is already represented by the facet columns.
delay_rates['Bar'] = ''

# Show the calculated rates
display(delay_rates)


# -------------------------------------------------------
# 5. Create the 2 x 2 FacetGrid
#
#                  UA              DL
#
# Departure       RATE            RATE
#
# Arrival         RATE            RATE
#
# -------------------------------------------------------

g = sns.FacetGrid(
    delay_rates,
    row='Delay_Type',
    col='OP_UNIQUE_CARRIER',
    row_order=['Departure', 'Arrival'],
    col_order=['UA', 'DL'],
    height=3.5,
    aspect=1.1,
    sharex=True,
    sharey=True
)


# -------------------------------------------------------
# 6. Draw ONE centered bar in each facet
# -------------------------------------------------------

g.map_dataframe(
    sns.barplot,
    x='Bar',
    y='Delay_Rate',
    width=0.55
)


# -------------------------------------------------------
# 7. Use the same percentage scale for all facets
# -------------------------------------------------------

g.set(
    ylim=(0, 100)
)


# -------------------------------------------------------
# 8. Labels
# -------------------------------------------------------

g.set_axis_labels(
    "",
    "Flights Delayed (%)"
)


# -------------------------------------------------------
# 9. Facet titles
# -------------------------------------------------------

g.set_titles(
    row_template="{row_name}",
    col_template="{col_name}"
)


# -------------------------------------------------------
# 10. Remove the unnecessary x-axis completely
# -------------------------------------------------------

for ax in g.axes.flat:
    ax.set_xticks([])
    ax.set_xlabel('')


# -------------------------------------------------------
# 11. Put the percentage above each bar
# -------------------------------------------------------

for ax in g.axes.flat:
    for container in ax.containers:
        ax.bar_label(
            container,
            fmt='%.1f%%',
            padding=3
        )


# -------------------------------------------------------
# 12. Overall title
# -------------------------------------------------------

g.fig.subplots_adjust(
    top=0.88,
    hspace=0.25
)

g.fig.suptitle(
    "Departure and Arrival Delay Rates: United vs Delta",
    fontsize=14
)


# -------------------------------------------------------
# 13. Display
# -------------------------------------------------------

plt.show()

<class 'ModuleNotFoundError'>: No module named 'pandas'

### Conclusions
### Departure Delay is the biggest factor in determining arrival delays
### FLL Airport experiences larger delays and variability than other airports
### Delta delay rate is higher than United Airlines

In [7]:
### Plan of Action
### The following plan of action is suggested to top management as a result of this analysis:
-- Examine ways to reduce departure delays, as this will have a high impact on overall delays (arrival delays)
-- Examine the reasons that FLL airport experiences considerably more delays, and also more variability of delays than at ATL and other airports.
-- Examine the reasons that Delta has significantly higher delay rates.


<class 'SyntaxError'>: invalid syntax (19407547.py, line 3)